## Descripción

Este notebook construye variables agregadas a nivel de cliente (`SK_ID_CURR`) a partir del historial mensual de créditos POS/Cash. La idea central es transformar una tabla transaccional o longitudinal, con varias observaciones por crédito previo y por mes, en una tabla de features lista para integrarse con otros insumos del modelo.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../../data')
df = pd.read_parquet(DATA_PATH/'pos_cash_balance.parquet')

## Construcción de features POS/Cash por cliente

Esta sección define `build_pos_cash_features`, una función que resume el comportamiento mensual de productos POS/Cash por `SK_ID_CURR`. El proceso crea variables binarias a nivel fila, identifica el estado más reciente de cada crédito previo y después agrega la información a nivel cliente.

Está capturando:

- **Actividad y cierre:** proporción de meses activos y completados.
- **Mora significativa:** presencia y máximo valor de `SK_DPD_DEF`.
- **Situación reciente:** cantidad de créditos todavía activos y cuotas futuras pendientes en el snapshot más cercano a la solicitud actual.

Estas variables resumen persistencia de obligaciones, historial de incumplimiento y carga futura. Son señales útiles porque el modelo necesita una fila por cliente, no una fila por mes o por crédito previo.

In [2]:
import pandas as pd
import numpy as np

def build_pos_cash_features(pos_cash_balance):
    pos = pos_cash_balance.copy()

    id_col = "SK_ID_CURR"
    prev_id_col = "SK_ID_PREV"
    status_col = "NAME_CONTRACT_STATUS"

    # VARIABLES DERIVADAS A NIVEL FILA

    pos["pos_is_active"] = pos[status_col].eq("Active").astype(int)
    pos["pos_is_completed"] = pos[status_col].eq("Completed").astype(int)

    # Mora significativa: con tolerancia
    pos["pos_has_dpd_def"] = pos["SK_DPD_DEF"].gt(0).astype(int)


    # SNAPSHOT MÁS RECIENTE POR CRÉDITO PREVIO
    # MONTHS_BALANCE más alto es el más cercano a la solicitud actual
    # Ejemplo: -1 es más reciente que -20

    recent_idx = (
        pos.groupby([id_col, prev_id_col])["MONTHS_BALANCE"]
        .idxmax()
    )

    recent_pos = pos.loc[recent_idx].copy()

    recent_pos["pos_recent_is_active"] = (
        recent_pos[status_col].eq("Active")
    ).astype(int)


    # FEATURES AGREGADAS POR SK_ID_CURR

    features = (
        pos.groupby(id_col)
        .agg(
            # Proporción de meses donde los créditos estuvieron activos
            pos_cash_active_rate=("pos_is_active", "mean"),

            # Proporción de meses donde los créditos aparecen completados
            pos_cash_completed_rate=("pos_is_completed", "mean"),

            # Frecuencia de meses con mora significativa
            pos_cash_dpd_def_positive_rate=("pos_has_dpd_def", "mean"),

            # Máxima mora significativa observada
            pos_cash_dpd_def_max=("SK_DPD_DEF", "max")
        )
    )


    # FEATURES DEL SNAPSHOT MÁS RECIENTE

    recent_features = (
        recent_pos.groupby(id_col)
        .agg(
            # Créditos POS/cash que seguían activos cerca de la solicitud actual
            pos_cash_recent_active_count=("pos_recent_is_active", "sum"),

            # Cuotas pendientes promedio en el snapshot más reciente
            pos_cash_recent_installments_future_mean=("CNT_INSTALMENT_FUTURE", "mean"),

            # Máximo número de cuotas pendientes en el snapshot más reciente
            pos_cash_recent_installments_future_max=("CNT_INSTALMENT_FUTURE", "max")
        )
    )

    features = features.join(recent_features)

    return features


pos_cash_features = build_pos_cash_features(df)

display(pos_cash_features.head())
print(pos_cash_features.shape)
print(pos_cash_features.columns.tolist())

,pos_cash_active_rate,pos_cash_completed_rate,pos_cash_dpd_def_positive_rate,pos_cash_dpd_def_max,pos_cash_recent_active_count,pos_cash_recent_installments_future_mean,pos_cash_recent_installments_future_max
SK_ID_CURR,,,,,,,
100001,0.777778,0.222222,0.111111,7,0,0.000000,0.0
100002,1.000000,0.000000,0.000000,0,1,6.000000,6.0
100003,0.928571,0.071429,0.000000,0,1,0.333333,1.0
100004,0.750000,0.250000,0.000000,0,0,0.000000,0.0
100005,0.818182,0.090909,0.000000,0,0,0.000000,0.0


(337252, 7)
['pos_cash_active_rate', 'pos_cash_completed_rate', 'pos_cash_dpd_def_positive_rate', 'pos_cash_dpd_def_max', 'pos_cash_recent_active_count', 'pos_cash_recent_installments_future_mean', 'pos_cash_recent_installments_future_max']
